In [ ]:
import os
import warnings 
import random 
from zipfile import ZipFile 
from pathlib import Path 
import copy
import glob 

import pandas as pd 
import sklearn 
import numpy as np

import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 
from torchinfo import summary

import torch
from torch import nn 
from torch .utils.data import DataLoader, Dataset 
import torch.nn.functional as F
from torchvision import transforms 
from torchvision.transforms import v2
import cv2

from sklearn.model_selection import GroupKFold

import timm

warnings.filterwarnings("ignore")

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
DATA_DIR = "/kaggle/input/rsna-knee-abnormality-detection"

In [ ]:
device = torch.device("mps"
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print(f"Device: '{device}'")

SEED = 2026
N_SPLITS = 5
BATCH_SIZE = 64
EPOCHS = 25
LR = 1e-3

Device: 'mps'


In [10]:
def seed_everything(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic=(True)
    torch.backends.cudnn.benchmark=(False)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(SEED)

In [ ]:
TARGET_COLS = [
    'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
    'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
    'Synovitis', "Baker's", 'Contusion', 'Fracture'
]

In [ ]:
class RSNADataset(Dataset):

    def __init__(self, df, img_size=224, is_train=True):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size 
        self.is_train = is_train 

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row["StudyInstanceUID"]
        study_path = os.path.join(DATA_DIR, "train_series", study_id)
        dcm_files = sorted(glob.glob(os.path.join(study_path, "**", "*.dcm"), recursive=True))

        if len(dcm_files) > 0:
            mid_idx = len(dcm_files) // 2
            selected_file = dcm_files[mid_idx]

            try:
                dcm_data = pydicom.dcmread(selected_file)
                image = dcm_data.pixel_array.astype(np.float32)

                if image.max() != image.min():
                    image = (image - image.min()) / (image.max() - image.min()) * 255.0 
                image = image.astype(np.uint8)
                image = cv2.resize(image, (self.img_size, self.img_size))
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            except Exception:
                image = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            image = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)

        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))
        image = torch.tensor(image, dtype=torch.float32)

        if self.is_train:
            labels = row[TARGET_COLS].values.astype(np.float32)
            labels = np.nan_to_num(labels, nan=0.0)
            labels = torch.tensor(labels, dtype=torch.float32)
            return image, labels 
        return image 

In [ ]:
import pydicom

# Укажите путь к любому реальному .dcm файлу из вашего датасета
sample_dcm_path = "train_series/ВАШ_STUDY_ID/ВАШ_SERIES_ID/ВАШ_FILE_ID.dcm"

# Читаем DICOM файл
dicom_data = pydicom.dcmread(sample_dcm_path)

# Достаем пиксели в виде numpy-массива
image_array = dicom_data.pixel_array

print(f"Размер картинки: {image_array.shape}")
print(f"Тип данных: {image_array.dtype}")

# Быстрый вывод картинки на экран для проверки
plt.imshow(image_array, cmap='gray')
plt.axis('off')
plt.show()